# Exercise 3 (VAE) + Bonus 4 (frozen ResNet-18)

BRISC classification MRI. Labels are not used in the VAE loss.
Decoder matches the Exercise 2 ConvAE decoder.


In [1]:
import math, json, os
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.neighbors import KNeighborsClassifier

from brisc import build, CLASS_TRAIN

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))
OUT = Path("outputs/task3")
PIXELS = 128 * 128
LATENT_DIM, BATCH = 8, 4  # test sizes


device: cuda


NVIDIA A100-SXM4-80GB


## Data — same split as Exercise 2 (seed 42, 80/20)


In [2]:
ds, train_data, val_data, train_loader, val_loader, encode_loader = build()
print("classes", ds.classes)
print("images", len(ds), "->", len(train_data), "train /", len(val_data), "val")
x, y = next(iter(train_loader))
print("batch", tuple(x.shape), float(x.min()), float(x.max()))


classes ['glioma', 'meningioma', 'no_tumor', 'pituitary']
images 5000 -> 4000 train / 1000 val


batch (64, 1, 128, 128) 0.0 1.0


## ConvVAE — encoder emits (mu, logvar); decoder = Exercise 2


In [3]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.ReLU(),
        )
        hidden = 128 * 8 * 8
        self.fc_mu = nn.Linear(hidden, latent_dim)
        self.fc_logvar = nn.Linear(hidden, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 4, stride=2, padding=1), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x).flatten(start_dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        return self.decoder(self.from_latent(z).view(-1, 128, 8, 8))

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar


def kl_divergence(mu, logvar):
    return 0.5 * torch.sum(mu.pow(2) + logvar.exp() - logvar - 1.0, dim=-1)


def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    recon = (x_hat - x).pow(2).flatten(1).sum(1).mean()  # sum pixels, mean batch
    kl = kl_divergence(mu, logvar).mean()
    return recon + beta * kl, recon, kl


@torch.no_grad()
def interpolate(model, x_a, x_b, steps=9):
    model.eval()
    dev = next(model.parameters()).device
    if x_a.dim() == 3: x_a = x_a.unsqueeze(0)
    if x_b.dim() == 3: x_b = x_b.unsqueeze(0)
    mu_a, _ = model.encode(x_a.to(dev))
    mu_b, _ = model.encode(x_b.to(dev))
    frames = []
    for i in range(steps):
        t = i / (steps - 1)
        frames.append(model.decode((1 - t) * mu_a + t * mu_b).cpu())
    return torch.cat(frames, 0)


@torch.no_grad()
def sample_from_prior(model, n=8):
    model.eval()
    dev = next(model.parameters()).device
    return model.decode(torch.randn(n, model.latent_dim, device=dev)).cpu()


## Unit tests (synthetic tensors, no dataset)


In [4]:
_TESTS = {}
def test(fn):
    _TESTS[fn.__name__] = fn
    return fn
def close(a, b, rel=1e-5, abs_tol=1e-8):
    return math.isclose(float(a), float(b), rel_tol=rel, abs_tol=abs_tol)
def run(prefix=""):
    names = sorted(n for n in _TESTS if n.startswith(prefix))
    failures = []
    for name in names:
        try:
            _TESTS[name]()
        except Exception as e:
            failures.append((name, f"{type(e).__name__}: {e}"))
    print(f"{len(names)-len(failures)}/{len(names)} passed")
    for n, e in failures:
        print("  FAIL", n, e)
    return not failures

@test
def test_construct_records_latent_dim():
    assert ConvVAE(2).latent_dim == 2 and ConvVAE(128).latent_dim == 128
@test
def test_construct_has_trainable_parameters():
    assert any(p.requires_grad for p in ConvVAE(LATENT_DIM).parameters())
@test
def test_encode_returns_two_tensors():
    out = ConvVAE(LATENT_DIM).encode(torch.rand(BATCH,1,128,128))
    assert isinstance(out, tuple) and len(out)==2
@test
def test_encode_shapes():
    mu, lv = ConvVAE(LATENT_DIM).encode(torch.rand(BATCH,1,128,128))
    assert mu.shape == lv.shape == (BATCH, LATENT_DIM)
@test
def test_encode_mu_and_logvar_are_distinct():
    mu, lv = ConvVAE(LATENT_DIM).encode(torch.rand(BATCH,1,128,128))
    assert not torch.allclose(mu, lv)
@test
def test_encode_different_inputs_differ():
    m = ConvVAE(LATENT_DIM)
    assert not torch.allclose(m.encode(torch.zeros(1,1,128,128))[0], m.encode(torch.ones(1,1,128,128))[0])
@test
def test_reparam_shape_matches_mu():
    m, mu = ConvVAE(LATENT_DIM), torch.zeros(BATCH, LATENT_DIM)
    assert m.reparameterize(mu, torch.zeros_like(mu)).shape == mu.shape
@test
def test_reparam_is_stochastic():
    m, mu = ConvVAE(LATENT_DIM), torch.zeros(BATCH, LATENT_DIM)
    lv = torch.zeros_like(mu)
    assert not torch.allclose(m.reparameterize(mu,lv), m.reparameterize(mu,lv))
@test
def test_reparam_collapses_to_mu_when_variance_vanishes():
    m = ConvVAE(LATENT_DIM)
    mu = torch.randn(BATCH, LATENT_DIM)
    assert torch.allclose(m.reparameterize(mu, torch.full_like(mu, -50)), mu, atol=1e-4)
@test
def test_reparam_empirical_moments():
    torch.manual_seed(0)
    m = ConvVAE(LATENT_DIM)
    n=20000
    z = m.reparameterize(torch.full((n,1), 3.), torch.full((n,1), math.log(4.)))
    assert close(z.mean(), 3, rel=0, abs_tol=0.05) and close(z.std(), 2, rel=0, abs_tol=0.05)
@test
def test_reparam_gradients_reach_both_arguments():
    m = ConvVAE(LATENT_DIM)
    mu = torch.zeros(BATCH, LATENT_DIM, requires_grad=True)
    lv = torch.zeros(BATCH, LATENT_DIM, requires_grad=True)
    m.reparameterize(mu, lv).sum().backward()
    assert mu.grad is not None and lv.grad is not None
@test
def test_reparam_preserves_dtype_and_device():
    m = ConvVAE(LATENT_DIM)
    mu = torch.zeros(BATCH, LATENT_DIM, dtype=torch.float64)
    z = m.reparameterize(mu, torch.zeros_like(mu))
    assert z.dtype == mu.dtype and z.device == mu.device
@test
def test_decode_shape():
    assert ConvVAE(LATENT_DIM).decode(torch.randn(BATCH, LATENT_DIM)).shape == (BATCH,1,128,128)
@test
def test_decode_output_is_bounded():
    out = ConvVAE(LATENT_DIM).decode(torch.randn(BATCH, LATENT_DIM)*10)
    assert out.min() >= 0 and out.max() <= 1
@test
def test_decode_accepts_a_prior_sample():
    out = ConvVAE(LATENT_DIM).decode(torch.randn(1, LATENT_DIM))
    assert out.shape == (1,1,128,128) and torch.isfinite(out).all()
@test
def test_forward_returns_three_values():
    assert len(ConvVAE(LATENT_DIM)(torch.rand(BATCH,1,128,128))) == 3
@test
def test_forward_shapes():
    x = torch.rand(BATCH,1,128,128)
    xh, mu, lv = ConvVAE(LATENT_DIM)(x)
    assert xh.shape == x.shape and mu.shape == (BATCH, LATENT_DIM)
@test
def test_forward_reconstruction_is_bounded():
    xh, _, _ = ConvVAE(LATENT_DIM)(torch.rand(BATCH,1,128,128))
    assert xh.min() >= 0 and xh.max() <= 1
@test
def test_forward_gradients_reach_the_encoder():
    m = ConvVAE(LATENT_DIM)
    m(torch.rand(BATCH,1,128,128))[0].sum().backward()
    assert any(p.grad is not None and p.grad.abs().sum()>0 for p in m.parameters())
@test
def test_kl_returns_one_value_per_sample():
    assert kl_divergence(torch.zeros(BATCH,LATENT_DIM), torch.zeros(BATCH,LATENT_DIM)).shape == (BATCH,)
@test
def test_kl_zero_at_the_prior():
    out = kl_divergence(torch.zeros(BATCH,LATENT_DIM), torch.zeros(BATCH,LATENT_DIM))
    assert torch.allclose(out, torch.zeros(BATCH), atol=1e-6)
@test
def test_kl_never_negative():
    torch.manual_seed(0)
    out = kl_divergence(torch.randn(64,LATENT_DIM)*3, torch.randn(64,LATENT_DIM)*2)
    assert (out >= -1e-6).all()
@test
def test_kl_shifted_mean_hand_computed():
    assert close(kl_divergence(torch.tensor([[1.]]), torch.tensor([[0.]])).item(), 0.5, rel=0, abs_tol=1e-5)
@test
def test_kl_widened_variance_hand_computed():
    assert close(kl_divergence(torch.tensor([[0.]]), torch.tensor([[1.]])).item(), 0.5*(math.e-2), rel=0, abs_tol=1e-5)
@test
def test_kl_narrowing_is_penalized():
    mu = torch.zeros(1,1)
    assert kl_divergence(mu, torch.tensor([[-4.]])).item() > kl_divergence(mu, torch.tensor([[0.]])).item()
@test
def test_kl_sums_over_latent_dimensions():
    one = kl_divergence(torch.tensor([[1.]]), torch.tensor([[0.]])).item()
    four = kl_divergence(torch.ones(1,4), torch.zeros(1,4)).item()
    assert close(four, 4*one, rel=0, abs_tol=1e-5)
@test
def test_kl_gradients_flow():
    mu = torch.randn(BATCH,LATENT_DIM, requires_grad=True)
    lv = torch.randn(BATCH,LATENT_DIM, requires_grad=True)
    kl_divergence(mu, lv).sum().backward()
    assert mu.grad is not None and lv.grad is not None

def _lp():
    torch.manual_seed(0)
    return torch.rand(BATCH,1,128,128), torch.rand(BATCH,1,128,128), torch.randn(BATCH,LATENT_DIM), torch.randn(BATCH,LATENT_DIM)
@test
def test_loss_returns_three_scalars():
    assert all(v.dim()==0 for v in vae_loss(*_lp()))
@test
def test_loss_total_is_the_weighted_sum():
    t,r,k = vae_loss(*_lp(), beta=2.5)
    assert close(t.item(), r.item()+2.5*k.item())
@test
def test_loss_beta_zero_drops_the_kl_term():
    t,r,_ = vae_loss(*_lp(), beta=0)
    assert close(t.item(), r.item())
@test
def test_loss_perfect_reconstruction_is_free():
    x = torch.rand(BATCH,1,128,128)
    _, rec, _ = vae_loss(x,x, torch.zeros(BATCH,LATENT_DIM), torch.zeros(BATCH,LATENT_DIM))
    assert close(rec.item(), 0, rel=0, abs_tol=1e-6)
@test
def test_loss_sums_over_pixels():
    mu, lv = torch.zeros(1,LATENT_DIM), torch.zeros(1,LATENT_DIM)
    big = vae_loss(torch.full((1,1,128,128),0.1), torch.zeros(1,1,128,128), mu, lv)[1]
    small = vae_loss(torch.full((1,1,64,64),0.1), torch.zeros(1,1,64,64), mu, lv)[1]
    assert close(big.item(), 4*small.item(), rel=1e-4)
@test
def test_loss_invariant_to_batch_size():
    torch.manual_seed(0)
    x, xh, mu, lv = torch.rand(1,1,128,128), torch.rand(1,1,128,128), torch.randn(1,LATENT_DIM), torch.randn(1,LATENT_DIM)
    s = vae_loss(xh,x,mu,lv)[0].item()
    d = vae_loss(xh.repeat(2,1,1,1), x.repeat(2,1,1,1), mu.repeat(2,1), lv.repeat(2,1))[0].item()
    assert close(d,s)
@test
def test_loss_gradients_reach_the_model():
    m = ConvVAE(LATENT_DIM); x = torch.rand(BATCH,1,128,128)
    xh,mu,lv = m(x); vae_loss(xh,x,mu,lv)[0].backward()
    assert any(p.grad is not None and p.grad.abs().sum()>0 for p in m.parameters())
@test
def test_end_to_end_loss_decreases():
    torch.manual_seed(0)
    m = ConvVAE(LATENT_DIM); x = torch.rand(BATCH,1,128,128)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    first=None; total=None
    for step in range(30):
        xh,mu,lv = m(x); total,_,_ = vae_loss(xh,x,mu,lv)
        opt.zero_grad(); total.backward(); opt.step()
        if step==0: first=total.item()
    assert total.item() < first

ok = run("test_")
print("Exercise 3 tests:", "ALL GREEN" if ok else "FAILED")
assert ok


35/35 passed
Exercise 3 tests: ALL GREEN


## Load trained VAEs (25 epochs, beta 0.01 and 1.0) and show proofs


In [5]:
def load_vae(tag):
    m = ConvVAE(64).to(device)
    path = OUT / f"best_{tag}.pt"
    m.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    m.eval()
    return m

vae_lo = load_vae("b001")
vae_hi = load_vae("b1")
metrics = json.loads((OUT / "metrics.json").read_text())
print(json.dumps({k: metrics[k] for k in ("exercise2","b001","b1")}, indent=2))

# glioma (0) -> pituitary (3), same protocol as Exercise 2
ia = ib = None
for i in range(len(val_data)):
    lab = int(val_data[i][1])
    if lab==0 and ia is None: ia=i
    if lab==3 and ib is None: ib=i
    if ia is not None and ib is not None: break
xa, xb = val_data[ia][0], val_data[ib][0]
frames = interpolate(vae_hi, xa, xb, 9)
priors = sample_from_prior(vae_hi, 8)

def show_strip(batch, title, names=None):
    n = batch.shape[0]
    fig, axes = plt.subplots(1, n, figsize=(1.5*n, 2.2))
    for i, ax in enumerate(axes):
        ax.imshow(batch[i,0].numpy(), cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
        ax.set_title(names[i] if names else f"{i/(n-1):.2f}", fontsize=7)
    fig.suptitle(title)
    plt.show()

show_strip(frames, f"VAE interpolation glioma→pituitary (val {ia}→{ib}), posterior means")
show_strip(priors, "Prior samples  z~N(0,I)  — generated, not reconstructed",
           [f"z{i}" for i in range(8)])

print("Unit reminder: VAE recon is SUM over 16384 pixels.")
print(f"  beta 0.01  {metrics['b001']['val_recon_sum_pixels']:.2f} sum  = {metrics['b001']['val_recon_per_pixel']:.5f}/px")
print(f"  beta 1.0   {metrics['b1']['val_recon_sum_pixels']:.2f} sum  = {metrics['b1']['val_recon_per_pixel']:.5f}/px")
print(f"  Ex2 AE d64 {metrics['exercise2']['d64_val_mse_per_pixel']:.5f}/px")


{
  "exercise2": {
    "d64_val_mse_per_pixel": 0.0061,
    "d64_val_mse_sum_pixels": 99.9424,
    "knn_acc": 0.875,
    "silhouette": 0.0128
  },
  "b001": {
    "beta": 0.01,
    "best_val_total": 109.80886926269531,
    "last_mu_abs": 0.8370752358436584,
    "last_sigma": 0.028855342999100687,
    "val_recon_sum_pixels": 107.43731341552734,
    "val_recon_per_pixel": 0.0065574532113969325
  },
  "b1": {
    "beta": 1.0,
    "best_val_total": 168.4455808105469,
    "last_mu_abs": 0.3772077045440674,
    "last_sigma": 0.726150622844696,
    "val_recon_sum_pixels": 138.18594848632813,
    "val_recon_per_pixel": 0.008434200957417488
  }
}


Unit reminder: VAE recon is SUM over 16384 pixels.
  beta 0.01  107.44 sum  = 0.00656/px
  beta 1.0   138.19 sum  = 0.00843/px
  Ex2 AE d64 0.00610/px


## Bonus 4 — frozen ImageNet ResNet-18

Same split, k=5. Only the extractor changes.
Preprocess: 3 channels, 224px, ImageNet mean/std. Weights frozen.


In [6]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_feature_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def build_frozen_resnet():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Identity()
    for p in m.parameters():
        p.requires_grad = False
    return m.eval()

@torch.no_grad()
def extract_features(model, loader, device="cpu"):
    model = model.to(device).eval()
    feats, labs = [], []
    for x, y in loader:
        feats.append(model(x.to(device)).cpu().numpy())
        labs.append(y.cpu().numpy() if torch.is_tensor(y) else np.asarray(y))
    return np.concatenate(feats), np.concatenate(labs)

def reduce_dimensions(train_f, val_f, n_components=64):
    pca = PCA(n_components=n_components)
    return pca.fit_transform(train_f), pca.transform(val_f)

def evaluate_features(tr_x, tr_y, va_x, va_y, n_neighbors=5):
    knn = KNeighborsClassifier(n_neighbors=n_neighbors)
    knn.fit(tr_x, tr_y)
    pred = knn.predict(va_x)
    return {
        "accuracy": float(accuracy_score(va_y, pred)),
        "silhouette": float(silhouette_score(va_x, va_y)),
        "report": classification_report(va_y, pred, zero_division=0),
    }


In [7]:
# Bonus tests (offline + ResNet weights)
def _gray(v=128):
    return Image.fromarray(np.full((128,128), v, dtype=np.uint8), "L")
@test
def test_transform_output_shape():
    assert build_feature_transform()(_gray()).shape == (3,224,224)
@test
def test_transform_size_is_configurable():
    assert build_feature_transform(128)(_gray()).shape == (3,128,128)
@test
def test_transform_replicates_grayscale_across_channels():
    out = build_feature_transform()(_gray())
    r = torch.stack([out[c]*IMAGENET_STD[c]+IMAGENET_MEAN[c] for c in range(3)])
    assert torch.allclose(r[0], r[1], atol=1e-5) and torch.allclose(r[1], r[2], atol=1e-5)
@test
def test_transform_applies_imagenet_normalization():
    out = build_feature_transform()(_gray(255))
    for c in range(3):
        assert close(out[c].mean(), (1-IMAGENET_MEAN[c])/IMAGENET_STD[c], rel=0, abs_tol=1e-3)
@test
def test_resnet_no_parameter_requires_gradient():
    assert all(not p.requires_grad for p in build_frozen_resnet().parameters())
@test
def test_resnet_is_in_evaluation_mode():
    assert not build_frozen_resnet().training
@test
def test_resnet_output_width_is_512():
    assert build_frozen_resnet()(torch.randn(2,3,224,224)).shape == (2,512)
@test
def test_resnet_classifier_head_is_gone():
    assert build_frozen_resnet()(torch.randn(1,3,224,224)).shape[1] != 1000
@test
def test_resnet_weights_are_pretrained_not_random():
    x = torch.randn(2,3,224,224)
    assert torch.allclose(build_frozen_resnet()(x), build_frozen_resnet()(x), atol=1e-5)
@test
def test_resnet_different_inputs_give_different_features():
    m = build_frozen_resnet()
    assert not torch.allclose(m(torch.zeros(1,3,224,224)), m(torch.ones(1,3,224,224)))

class _Probe(nn.Module):
    def __init__(self):
        super().__init__(); self.grad_was_enabled=None
    def forward(self, x):
        self.grad_was_enabled = torch.is_grad_enabled()
        return x.flatten(1)[:, :512]
def _stub():
    return DataLoader(TensorDataset(torch.randn(10,3,224,224), torch.tensor([0,1,2,3,0,1,2,3,0,1])), batch_size=4, shuffle=False)
@test
def test_extract_shapes_and_types():
    f,l = extract_features(_Probe().eval(), _stub())
    assert f.shape==(10,512) and l.shape==(10,) and isinstance(f, np.ndarray)
@test
def test_extract_preserves_label_order():
    _, l = extract_features(_Probe().eval(), _stub())
    assert list(l)==[0,1,2,3,0,1,2,3,0,1]
@test
def test_extract_builds_no_graph():
    p=_Probe().eval(); extract_features(p, _stub())
    assert p.grad_was_enabled is False
@test
def test_reduce_output_shapes():
    tr,va = reduce_dimensions(np.random.randn(137,512), np.random.randn(41,512), 64)
    assert tr.shape==(137,64) and va.shape==(41,64)
@test
def test_reduce_structure_survives():
    rng=np.random.default_rng(0)
    stacked=np.vstack([rng.normal(0,1,(100,512)), rng.normal(20,1,(100,512))])
    red,_=reduce_dimensions(stacked, stacked[:5], 8)
    assert np.linalg.norm(red[:100].mean(0)-red[100:].mean(0)) > 5*red[:100].std(0).mean()
@test
def test_reduce_shared_projection():
    rng=np.random.default_rng(1)
    train=rng.normal(size=(80,512)); val=np.vstack([train[3], rng.normal(size=(19,512))])
    tr,va=reduce_dimensions(train,val,16)
    assert np.allclose(tr[3], va[0], atol=1e-6)
@test
def test_reduce_validation_is_not_refitted():
    rng=np.random.default_rng(2)
    train=rng.normal(size=(80,512)); val=rng.normal(size=(20,512))
    _,a=reduce_dimensions(train,val,16); _,b=reduce_dimensions(train,val+50,16)
    assert np.linalg.norm(b.mean(0)-a.mean(0)) > 1.0
def _sep():
    rng=np.random.default_rng(0); C=np.array([[0,0],[30,0],[0,30],[30,30]], float)
    trx,try_,vax,vay=[],[],[],[]
    for i,c in enumerate(C):
        trx.append(rng.normal(c,1,(50,2))); try_ += [i]*50
        vax.append(rng.normal(c,1,(20,2))); vay += [i]*20
    return np.vstack(trx), np.array(try_), np.vstack(vax), np.array(vay)
@test
def test_evaluate_returns_expected_keys():
    assert set(evaluate_features(*_sep()))=={"accuracy","silhouette","report"}
@test
def test_evaluate_separable_data_scores_high():
    assert evaluate_features(*_sep())["accuracy"]>0.95
@test
def test_evaluate_separable_data_has_high_silhouette():
    assert evaluate_features(*_sep())["silhouette"]>0.7
@test
def test_evaluate_random_features_score_near_chance():
    rng=np.random.default_rng(0)
    r=evaluate_features(rng.normal(size=(400,16)), np.tile([0,1,2,3],100),
                        rng.normal(size=(200,16)), np.tile([0,1,2,3],50))
    assert r["accuracy"]<0.40
@test
def test_evaluate_neighbor_count_is_honored():
    assert evaluate_features(*_sep(), n_neighbors=1)["accuracy"]>0.9

ok2 = run("test_transform") and run("test_resnet") and run("test_extract") and run("test_reduce") and run("test_evaluate")
print("Bonus tests:", "ALL GREEN" if ok2 else "FAILED")
assert ok2


4/4 passed


6/6 passed


3/3 passed


4/4 passed


5/5 passed
Bonus tests: ALL GREEN


## Bonus 4 numbers — same split, k=5 (from GPU run)


In [8]:
r = metrics["resnet"]
print(f"{'repr':22s} {'width':>5s} {'5-NN':>8s} {'sil':>8s}")
print(f"{'AE d64 (Ex2)':22s} {64:5d} {metrics['exercise2']['knn_acc']:8.4f} {metrics['exercise2']['silhouette']:8.4f}")
print(f"{'ResNet-18 frozen':22s} {512:5d} {r['width_512']['accuracy']:8.4f} {r['width_512']['silhouette']:8.4f}")
print(f"{'ResNet-18 → PCA':22s} {64:5d} {r['width_64']['accuracy']:8.4f} {r['width_64']['silhouette']:8.4f}")
print()
print("ResNet 512 report")
print(r["report_512"])

img = plt.imread(OUT / "resnet_pca_tsne.png")
plt.figure(figsize=(12,5)); plt.imshow(img); plt.axis("off"); plt.title("PCA / t-SNE of frozen ResNet-18 val features"); plt.show()


repr                   width     5-NN      sil
AE d64 (Ex2)              64   0.8750   0.0128
ResNet-18 frozen         512   0.9040   0.0299
ResNet-18 → PCA           64   0.9000   0.0400

ResNet 512 report
              precision    recall  f1-score   support

           0       0.87      0.86      0.87       206
           1       0.84      0.85      0.84       267
           2       0.97      0.93      0.95       239
           3       0.93      0.97      0.95       288

    accuracy                           0.90      1000
   macro avg       0.90      0.90      0.90      1000
weighted avg       0.90      0.90      0.90      1000



## Short answers

**KL.** Penalises off-centre means, too-wide variance, and collapse to a point. AE codes are isolated points → blurry interpolations. VAE samples local Gaussians so the decoder sees neighbourhoods.

**Beta.** Low beta ≈ AE recon; high beta ≈ stronger prior, worse recon. Compare the two runs; watch μ/σ for collapse.

**Generated images.** Decoder samples only — not clinical data.

**ResNet control.** Frozen ImageNet features, same split/k. If ResNet kNN is higher, classes are separable and the AE/VAE gap is the objective.
